# Industry Risk Analysis: Methodology and Output Map

This notebook is the guided methodology reference for the project. It follows the same logic as `METHODOLOGY.md` and `data/methodology.md`, but presents the project in a notebook format for reviewers who want a structured walkthrough rather than a long markdown file.

It covers:
- the public datasets
- the stage-by-stage pipeline logic
- the main equations and score logic
- every output table referenced in `data/methodology.md`

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

pd.options.display.max_columns = 100

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src').exists() and (candidate / 'output' / 'tables').exists():
            return candidate
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import PUBLIC_SOURCE_URLS, PROCESSED_DIR

TABLES = REPO_ROOT / 'output' / 'tables'
dataset_register = pd.DataFrame([
    ['ABS-AI', 'ABS Australian Industry 2023-24', 'Annual sector sales, employment, wages, EBITDA, and operating profit', 'Foundation, margins, sales growth, portfolio proxy'],
    ['ABS-BI-22', 'ABS Business Indicators: Gross Operating Profit / Sales Ratio', 'Quarterly profitability ratio by industry', 'Margin level and margin trend'],
    ['ABS-BI-23', 'ABS Business Indicators: Inventories / Sales Ratio', 'Quarterly inventory intensity by industry', 'Inventory days estimate and stock-build risk'],
    ['ABS-LF', 'ABS Labour Force by Industry', 'Monthly employment trend by industry', 'Employment-growth signal'],
    ['ABS-BA', 'ABS Building Approvals (Non-Residential)', 'Monthly non-residential approvals by building type', 'Demand proxy for selected sectors'],
    ['RBA-F1', 'RBA Cash Rate', 'Official cash-rate time series', 'Rate backdrop and pricing reference'],
    ['PTRS', 'Payment Times Reporting Scheme publications', 'Public payment timing by ANZSIC division', 'AR and AP timing proxy'],
], columns=['dataset_id', 'dataset', 'plain_english_meaning', 'main_use'])

display(dataset_register)
display(pd.DataFrame(PUBLIC_SOURCE_URLS.items(), columns=['source_key', 'url']))

## Pipeline Logic

### Stage 1: Foundation
- Builds four structural scores and takes their mean as `classification_risk_score`.

### Stage 2: Macro View
- Builds employment, margin, inventory, demand, and rate signals.
- Main formula:
  ```text
  inventory_days_est = inventories_to_sales_ratio x 91.25 / estimated_cogs_to_sales_ratio
  
  estimated_cogs_to_sales_ratio = clip(1 - margin_ratio, 0.45, 0.95)
  ```
- Main sector score:
  ```text
  industry_base_risk_score = 55% x classification_risk_score + 45% x macro_risk_score
  ```

### Stage 3: Benchmarks
- Builds AR, AP, inventory, leverage, and coverage benchmark fields.
- PTRS is used as the primary AR/AP timing source when available.

### Stage 4: Working Capital
- Builds AR/AP/inventory/CCC metrics and interpretation overlays.
- Main formula:
  ```text
  cash_conversion_cycle_benchmark_days = ar_days_benchmark + inventory_days_benchmark - ap_days_benchmark
  ```

### Stage 5: Bottom-Up
- Generates one synthetic borrower archetype per industry.

### Stage 6: Scorecard
- Main formula:
  ```text
  final_industry_risk_score = 35% x classification + 30% x macro + 35% x bottom_up
  ```

### Stage 7: Credit Application
- Pricing formula:
  ```text
  all_in_rate = cash_rate + base_margin + industry_loading
  ```
- Portfolio proxy formula:
  ```text
  current_exposure_pct = (70% x sales_share + 30% x employment_share) x 100
  ```

### Stage 8 and Stage 9
- Adds appetite, stress, ESG, watchlist, workbook, chart, and PDF outputs.

In [ ]:
table_registry = pd.DataFrame([
    ['Output Table 1', 'data/processed/industry_classification_foundation.csv', 'Structural classification scores'],
    ['Output Table 2', 'data/processed/industry_macro_view_public_signals.csv', 'Public macro signals and base risk'],
    ['Output Table 3', 'output/tables/industry_base_risk_scorecard.csv', 'Ranked sector base risk view'],
    ['Output Table 4', 'output/tables/industry_public_benchmarks.csv', 'Public benchmark pack'],
    ['Output Table 5', 'output/tables/industry_generated_benchmarks.csv', 'Generated benchmark proxies'],
    ['Output Table 5A', 'output/tables/industry_working_capital_risk_metrics.csv', 'Sector working-capital and PD/LGD overlays'],
    ['Output Table 5B', 'output/tables/borrower_working_capital_risk_metrics.csv', 'Borrower working-capital overlays'],
    ['Output Table 6', 'data/processed/borrower_benchmark_comparison.csv', 'Borrower vs benchmark comparison'],
    ['Output Table 7', 'output/tables/borrower_industry_risk_scorecard.csv', 'Final borrower scorecard'],
    ['Output Table 8', 'output/tables/industry_portfolio_proxy.csv', 'Public-data portfolio proxy'],
    ['Output Table 9', 'output/tables/concentration_limits.csv', 'Concentration limits and utilisation'],
    ['Output Table 10', 'output/tables/pricing_grid.csv', 'Illustrative pricing grid'],
    ['Output Table 11', 'output/tables/policy_overlay.csv', 'Borrower policy overlay'],
    ['Output Table 12', 'output/tables/industry_credit_appetite_strategy.csv', 'Sector credit appetite strategy'],
    ['Output Table 13', 'output/tables/industry_stress_test_matrix.csv', 'Stress scenario matrix'],
    ['Output Table 14', 'output/tables/industry_esg_sensitivity_overlay.csv', 'Sector ESG overlay'],
    ['Output Table 15', 'output/tables/watchlist_triggers.csv', 'Sector watchlist triggers'],
], columns=['table_id', 'path', 'purpose'])

display(table_registry)

In [ ]:
foundation = pd.read_csv(PROCESSED_DIR / 'industry_classification_foundation.csv')
macro = pd.read_csv(PROCESSED_DIR / 'industry_macro_view_public_signals.csv')
benchmarks = pd.read_csv(TABLES / 'industry_generated_benchmarks.csv')
industry_wc = pd.read_csv(TABLES / 'industry_working_capital_risk_metrics.csv')
borrower_compare = pd.read_csv(PROCESSED_DIR / 'borrower_benchmark_comparison.csv')
borrower_scorecard = pd.read_csv(TABLES / 'borrower_industry_risk_scorecard.csv')
portfolio_proxy = pd.read_csv(TABLES / 'industry_portfolio_proxy.csv')
concentration = pd.read_csv(TABLES / 'concentration_limits.csv')
pricing = pd.read_csv(TABLES / 'pricing_grid.csv')
appetite = pd.read_csv(TABLES / 'industry_credit_appetite_strategy.csv')
stress = pd.read_csv(TABLES / 'industry_stress_test_matrix.csv')
esg = pd.read_csv(TABLES / 'industry_esg_sensitivity_overlay.csv')
watchlist = pd.read_csv(TABLES / 'watchlist_triggers.csv')

display(foundation)
display(macro[['industry', 'classification_risk_score', 'macro_risk_score', 'industry_base_risk_score', 'industry_base_risk_level', 'inventory_days_est', 'inventory_stock_build_risk']].sort_values('industry_base_risk_score', ascending=False))
display(benchmarks[['industry', 'ar_days_benchmark', 'ap_days_benchmark', 'inventory_days_benchmark', 'debt_to_ebitda_benchmark', 'icr_benchmark', 'benchmark_origin']])
display(industry_wc[['industry', 'cash_conversion_cycle_benchmark_days', 'ar_collection_score', 'receivables_realisation_score', 'ap_supplier_stretch_score', 'working_capital_scorecard_overlay_score', 'working_capital_pd_overlay_score', 'working_capital_lgd_overlay_score']].sort_values('working_capital_pd_overlay_score', ascending=False))
display(borrower_compare[['borrower_name', 'industry', 'bottom_up_risk_score']])
display(borrower_scorecard)
display(pricing)
display(concentration.sort_values('utilisation_pct', ascending=False))
display(appetite)
display(stress.head(20))
display(esg)
display(watchlist)

## Interpretation

1. **Which sectors look structurally stronger or weaker?**
   This comes from the foundation and macro tables.

2. **What does that imply for a typical borrower?**
   This comes from the benchmark, borrower, and scorecard tables.

3. **How does working capital change that risk read?**
   This comes from the AR, AP, inventory, CCC, PD-overlay, and LGD-overlay tables.

4. **What would a portfolio team do with that information?**
   This comes from pricing, policy, concentration, appetite, stress, ESG, and watchlist outputs.